In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All ChromaDB data will persist here
CHROMA_PATH = "/content/drive/MyDrive/rag_day3/chroma_store"

import os
os.makedirs(CHROMA_PATH, exist_ok=True)
print(f"Storage path ready: {CHROMA_PATH}")

Mounted at /content/drive
Storage path ready: /content/drive/MyDrive/rag_day3/chroma_store


In [ ]:
!pip install -q chromadb sentence-transformers wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-learning-project/1.0'
)

# Change this topic to anything you want
TOPIC = "Transformer (deep learning architecture)"
page = wiki.page(TOPIC)

if not page.exists():
    print("Page not found. Try a different topic.")
else:
    raw_text = page.text
    print(f"Fetched: {len(raw_text)} characters")
    print(f"Preview:\n{raw_text[:300]}")

Fetched: 113029 characters
Preview:
In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. At each layer


In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    chunk_size: characters per chunk
    overlap: characters repeated between consecutive chunks
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text, chunk_size=500, overlap=50)
print(f"Total chunks: {len(chunks)}")
print(f"\n--- Chunk 0 ---")
print(chunks[0])
print(f"\n--- Chunk 1 (first 100 chars) ---")
print(chunks[1][:100])
print(f"\n--- Last 100 chars of Chunk 0 ---")
print(chunks[0][-100:])

Total chunks: 252

--- Chunk 0 ---
In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. At each layer, each token is then contextualized within the scope of the context window with other (unmasked) tokens via a parallel multi-head attention mechanism, allowing the signal for key tokens to be amplifie

--- Chunk 1 (first 100 chars) ---
 allowing the signal for key tokens to be amplified and less important tokens to be diminished. 
Tra

--- Last 100 chars of Chunk 0 ---
ens via a parallel multi-head attention mechanism, allowing the signal for key tokens to be amplifie


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 384-dim vectors, fast, free, no API key
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32
)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Each chunk → {embeddings.shape[1]}-dimensional vector")

# Sanity check: consecutive chunks should be semantically similar
from sklearn.metrics.pairwise import cosine_similarity

sim_consecutive = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
sim_unrelated = cosine_similarity([embeddings[0]], [embeddings[251]])[0][0]

print(f"\nCosine similarity: chunk 0 vs chunk 1 (consecutive): {sim_consecutive:.4f}")
print(f"Cosine similarity: chunk 0 vs chunk 251 (far apart):  {sim_unrelated:.4f}")
print("\nChunk 0 vs 1 should be higher than chunk 0 vs 251")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embedding matrix shape: (252, 384)
Each chunk → 384-dimensional vector

Cosine similarity: chunk 0 vs chunk 1 (consecutive): 0.6878
Cosine similarity: chunk 0 vs chunk 251 (far apart):  0.2701

Chunk 0 vs 1 should be higher than chunk 0 vs 251


In [ ]:
import chromadb

# PersistentClient writes to Drive — survives Colab restarts
client = chromadb.PersistentClient(path=CHROMA_PATH)

# get_or_create: safe to re-run without duplicating data
collection = client.get_or_create_collection(
    name="wiki_docs",
    metadata={"hnsw:space": "cosine"}
)

# Add in batches to avoid memory issues on large docs
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch_chunks = chunks[i:i+BATCH_SIZE]
    batch_embeddings = embeddings[i:i+BATCH_SIZE]
    collection.add(
        documents=batch_chunks,
        embeddings=batch_embeddings.tolist(),
        ids=[f"chunk_{j}" for j in range(i, i+len(batch_chunks))]
    )

print(f"Stored {collection.count()} chunks in ChromaDB")
print(f"Persisted at: {CHROMA_PATH}")

Stored 252 chunks in ChromaDB
Persisted at: /content/drive/MyDrive/rag_day3/chroma_store


In [ ]:
def retrieve(query, k=3):
    """
    Embed the query → search ChromaDB → return top-k chunks with scores.
    Distance: 0.0 = identical meaning, 1.0 = completely unrelated (cosine space)
    """
    # Step 1: embed the query using the same model used for documents
    query_embedding = model.encode([query]).tolist()

    # Step 2: search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    # Step 3: display results
    print(f"Query: '{query}'")
    print(f"Top {k} results:\n")

    for i, (doc, dist) in enumerate(
        zip(results['documents'][0], results['distances'][0])
    ):
        print(f"[{i+1}] Distance: {dist:.4f}")
        print(f"     {doc[:250]}")
        print()

    return results

# First real query
results = retrieve("What is the attention mechanism?", k=3)

Query: 'What is the attention mechanism?'
Top 3 results:

[1] Distance: 0.4092
     on, the scope of attention, or the range of token relationships captured by each attention head, can expand as tokens pass through successive layers. This allows the model to capture more complex and long-range dependencies in deeper layers. Many tra

[2] Distance: 0.4288
     direct objects. The computations for each attention head can be performed in parallel, which allows for fast processing. The outputs for the attention layer are concatenated to pass into the feedforward neural network layers.
Concretely, let the mult

[3] Distance: 0.4616
     GPUs. In 2016, decomposable attention applied a self-attention mechanism to feedforward networks, which are easy to parallelize, and achieved SOTA result in textual entailment with an order of magnitude fewer parameters than LSTMs. One of its authors



In [ ]:
# Query uses DIFFERENT words but same meaning
# This tests whether retrieval is truly semantic or just keyword matching
retrieve("How does the model decide what to focus on?", k=3)

Query: 'How does the model decide what to focus on?'
Top 3 results:

[1] Distance: 0.6393
     anguage models developed by Google AI

Notes
References


== Further reading ==

[2] Distance: 0.6404
     on, the scope of attention, or the range of token relationships captured by each attention head, can expand as tokens pass through successive layers. This allows the model to capture more complex and long-range dependencies in deeper layers. Many tra

[3] Distance: 0.6420
     efinitions of "relevance". Specifically, the query and key projection matrices, 
  
    
      
        
          W
          
            Q
          
        
      
    
    {\displaystyle W^{Q}}
  
 and 
  
    
      
        
          W
     



{'ids': [['chunk_251', 'chunk_96', 'chunk_94']],
 'embeddings': None,
 'documents': [['anguage models developed by Google AI\n\nNotes\nReferences\n\n\n== Further reading ==',
   'on, the scope of attention, or the range of token relationships captured by each attention head, can expand as tokens pass through successive layers. This allows the model to capture more complex and long-range dependencies in deeper layers. Many transformer attention heads encode relevance relations that are meaningful to humans. For example, some attention heads can attend mostly to the next word, while others mainly attend from verbs to their direct objects. The computations for each attentio',
   'efinitions of "relevance". Specifically, the query and key projection matrices, \n  \n    \n      \n        \n          W\n          \n            Q\n          \n        \n      \n    \n    {\\displaystyle W^{Q}}\n  \n and \n  \n    \n      \n        \n          W\n          \n            K\n          \n        \

In [9]:
# See how result quality changes as k increases
query = "encoder decoder architecture"
query_embedding = model.encode([query]).tolist()

print(f"Query: '{query}'\n")
print(f"{'k':<5} {'min_dist':<12} {'max_dist':<12} {'gap':<10}")
print("-" * 40)

for k in [1, 3, 5, 10]:
    r = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    distances = r['distances'][0]
    gap = max(distances) - min(distances)
    print(f"{k:<5} {min(distances):<12.4f} {max(distances):<12.4f} {gap:<10.4f}")

# Also print what k=1 returns vs what k=10's last result returns
print("\n--- Best result (k=1) ---")
r1 = collection.query(query_embeddings=query_embedding, n_results=1)
print(r1['documents'][0][0][:300])

print("\n--- Worst result (k=10, last chunk) ---")
r10 = collection.query(query_embeddings=query_embedding, n_results=10)
print(r10['documents'][0][-1][:300])

Query: 'encoder decoder architecture'

k     min_dist     max_dist     gap       
----------------------------------------
1     0.3608       0.3608       0.0000    
3     0.3608       0.4025       0.0417    
5     0.3608       0.4354       0.0746    
10    0.3608       0.4671       0.1063    

--- Best result (k=1) ---
 to attend by relative position."
In typical implementations, all operations are done over the real numbers, not the complex numbers, but since complex multiplication can be implemented as real 2-by-2 matrix multiplication, this is a mere notational difference.

Encoder–decoder (overview)
Like earli

--- Worst result (k=10, last chunk) ---
etc., autoregressively generating output text.

Full transformer architecture
Sublayers
Each encoder layer contains 2 sublayers: the self-attention and the feedforward network. Each decoder layer contains 3 sublayers: the causally masked self-attention, the cross-attention, and the feedforward netwo


In [10]:
# ChromaDB always returns k results — no "no match" response
# This is the most important failure mode to understand in RAG

irrelevant_query = "What is the recipe for biryani?"
query_embedding = model.encode([irrelevant_query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"Query: '{irrelevant_query}'")
print(f"\nChromaDB still returned {len(results['documents'][0])} chunks:\n")

for i, (doc, dist) in enumerate(
    zip(results['documents'][0], results['distances'][0])
):
    print(f"[{i+1}] Distance: {dist:.4f}")
    print(f"     {doc[:200]}")
    print()

# Now show what happens when you add a threshold filter
DISTANCE_THRESHOLD = 0.5

print("=" * 50)
print(f"With threshold filter (distance < {DISTANCE_THRESHOLD}):\n")

filtered = [
    (doc, dist)
    for doc, dist in zip(results['documents'][0], results['distances'][0])
    if dist < DISTANCE_THRESHOLD
]

if filtered:
    for doc, dist in filtered:
        print(f"Distance: {dist:.4f} | {doc[:200]}")
else:
    print("No chunks passed the threshold.")
    print("System should respond: 'I don't have information about this topic.'")

Query: 'What is the recipe for biryani?'

ChromaDB still returned 3 chunks:

[1] Distance: 0.7808
       ⋯
                
              
              
                
                  ⋮
                
                
                  ⋮
                
                
                  ⋮
    

[2] Distance: 0.7926
           
            V
          
        
      
    
    {\displaystyle W^{K},W^{V}}
  
, thus:

  
    
      
        
          MultiQueryAttention
        
        (
        Q
        ,
        K


[3] Distance: 0.8189
     1&0&1&2&\cdots \\-2&-1&0&1&\cdots \\-3&-2&-1&0&\cdots \\\vdots &\vdots &\vdots &\vdots &\ddots \\\end{pmatrix}}}
  
in other words, 
  
    
      
        
          B
          
            i
      

With threshold filter (distance < 0.5):

No chunks passed the threshold.
System should respond: 'I don't have information about this topic.'


# Week 4 Day 3 — Retrieval Pipeline

## What we built
A semantic search pipeline from scratch — no LangChain.

Takes a raw Wikipedia article → splits into chunks → converts each chunk into a vector → stores in ChromaDB on Google Drive → retrieves the most relevant chunks for any query using cosine similarity.

**Document used:** Wikipedia — "Transformer (deep learning architecture)"
- Raw size: 113,029 characters
- After chunking: 252 chunks
- Each chunk: 500 characters with 50-character overlap

---

## Why no LangChain
LangChain wraps every step of this pipeline behind abstractions. At this stage, understanding what each step does and why it exists matters more than convenience. Once the raw pipeline is clear, LangChain becomes a shortcut you can use consciously instead of magic you cannot debug.

---

## Pipeline

Raw Text

↓

Chunking (500 chars, 50 overlap)

↓

Embedding — all-MiniLM-L6-v2 → 384-dim vectors

↓

ChromaDB — PersistentClient on Google Drive (cosine space)

↓

Query → embed → cosine search → distance filter → top-k chunks

---

## What each step does and why

**Chunking**
Embedding models have a token limit. `all-MiniLM-L6-v2` caps at 256 tokens (~1000 characters). Feed it 113,000 characters and it silently truncates everything after the limit. Chunking splits the document into pieces that fit within this limit. The 50-character overlap ensures a sentence cut at a chunk boundary is not lost — it repeats at the start of the next chunk.

**Embedding**
Converts text into numbers. Each chunk becomes a 384-dimensional vector where semantically similar text lands geometrically close. "What is attention?" and "how tokens attend to each other" produce similar vectors even with no shared keywords. Same model must be used for both documents and queries — different models produce incompatible vector spaces.

**ChromaDB — PersistentClient**
Stores vectors + raw text on Google Drive. Survives Colab restarts. You embed once and query forever — no re-embedding on every session. `get_or_create_collection` makes it safe to re-run without duplicating data. `hnsw:space: cosine` tells ChromaDB to use cosine similarity instead of default L2 — cosine measures the angle between vectors (better for sentence embeddings), L2 measures straight-line distance (misses meaning when vector magnitudes differ).

**Similarity search**
ChromaDB compares the query vector against all 252 stored vectors and returns the k closest by cosine distance. Distance 0.0 = identical meaning. Distance 1.0 = completely unrelated. In practice, strong matches land below 0.5 for this model and dataset.

---

## Experiments & what we found

**Experiment A — semantic query with no shared keywords**
Query: "How does the model decide what to focus on?"
No words like "attention", "transformer", or "multi-head" in the query.

- Distances jumped to ~0.64 vs ~0.41 for technical queries
- `all-MiniLM-L6-v2` handles technical phrasing better than plain English
- chunk_251 (Wikipedia references section) appeared as top result — noise

**Experiment B — varying k**

| k | min_dist | max_dist | gap |
|---|----------|----------|-----|
| 1 | 0.3608 | 0.3608 | 0.0000 |
| 3 | 0.3608 | 0.4025 | 0.0417 |
| 5 | 0.3608 | 0.4354 | 0.0746 |
| 10 | 0.3608 | 0.4671 | 0.1063 |

Best match never changes regardless of k. Quality degrades 29% from rank 1 to rank 10. Every chunk retrieved goes into the LLM context — weak chunks at high k introduce noise. Start with k=3.

**Experiment C — irrelevant query**
Query: "What is the recipe for biryani?"

- ChromaDB returned 3 chunks anyway — distances 0.78, 0.79, 0.81
- It has no concept of "no match" — always returns k results
- All 3 returned chunks were LaTeX math notation (worst quality chunks in the database)
- After distance threshold filter (> 0.5): 0 chunks passed → correct fallback behavior

---

## Problems found

**Noise chunks enter retrieval**
The Wikipedia references section (`Notes`, `References`, `Further reading`) got chunked as normal content. The chunker treats all text equally — no awareness of what is meaningful vs structural. At high k or weak queries, these chunks surface as top results.
Fix: filter chunks below a minimum word count before storing, or strip known noise sections from raw text.

**LaTeX markup corrupts embeddings**
Wikipedia math notation (`{\displaystyle W^{Q}}`) leaked into chunks. The embedding model cannot make semantic sense of this, so these chunks become semantically empty — they end up closest to any query with no real match in the database.
Fix: clean raw text with regex before chunking to strip `{\displaystyle ...}` patterns.

**No rejection for out-of-scope queries**
ChromaDB always returns k results regardless of how irrelevant they are. Without a threshold, garbage flows silently into the LLM context.
Fix: filter by distance — if `distance > 0.5`, do not pass the chunk to the LLM. Respond with "I don't have information about this topic" instead.

**Plain English queries underperform**
Abstract queries ("how does the model decide what to focus on") return weaker results than technical queries ("what is the attention mechanism") even when asking about the same concept.
Fix: use a stronger embedding model (`all-mpnet-base-v2`) or add a reranker on top of retrieval in production.

---

## Key numbers

| Thing | Value |
|-------|-------|
| Raw document size | 113,029 characters |
| Total chunks | 252 |
| Chunk size | 500 characters |
| Overlap | 50 characters |
| Embedding dimensions | 384 |
| Embedding model | all-MiniLM-L6-v2 |
| Distance metric | Cosine |
| Distance threshold | 0.5 |
| Recommended k | 3 |

---

## What Day 4 builds on this
The filtered top-k chunks get passed to an LLM as context. The LLM reads only the retrieved chunks — not the full 113,000 character document. If retrieval is broken, LLM answers are broken regardless of how good the model is. Today's pipeline is the foundation everything else sits on.